# Spam Email Classification – Exploratory Data Analysis (EDA)

**PBL Review 3**

This notebook performs practical Exploratory Data Analysis (EDA) on the Spam Email Classification dataset.

### Objectives
- Load and inspect the dataset
- Identify the email/text and target columns
- Check missing values and duplicate records
- Analyze Spam vs Ham class distribution
- Analyze message length and word count
- Examine frequently occurring words
- Prepare observations for the machine learning implementation

In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("Libraries imported successfully.")

## 1. Locate and Load the Dataset

Place your dataset inside the `Dataset` folder of the project repository.

The code below searches common CSV filenames automatically. If your file has a different name, update `DATASET_PATH`.

In [ ]:
# Update this path if your dataset has a different filename.
DATASET_PATH = None

if DATASET_PATH is None:
    candidates = glob.glob("Dataset/*.csv") + glob.glob("dataset/*.csv") + glob.glob("*.csv")
    if not candidates:
        raise FileNotFoundError(
            "No CSV dataset found. Put your dataset inside the Dataset folder "
            "or set DATASET_PATH manually."
        )
    DATASET_PATH = candidates[0]

print("Dataset selected:", DATASET_PATH)

df = pd.read_csv(DATASET_PATH, encoding="utf-8", on_bad_lines="skip")
print("Dataset loaded successfully.")

In [ ]:
print("Rows and columns:", df.shape)
display(df.head())

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

## 2. Identify the Text and Label Columns

The project requires an email/message column and a Spam/Ham target column.

The following cell tries to identify them automatically. If the detected columns are incorrect, manually set `TEXT_COL` and `LABEL_COL`.

In [ ]:
# Automatic column detection
text_candidates = ["message", "text", "email", "email_text", "body", "content", "v2"]
label_candidates = ["label", "category", "class", "target", "spam", "v1"]

TEXT_COL = next((c for c in text_candidates if c in df.columns), None)
LABEL_COL = next((c for c in label_candidates if c in df.columns), None)

print("Detected text column:", TEXT_COL)
print("Detected label column:", LABEL_COL)

# If required, replace the values above manually, for example:
# TEXT_COL = "message"
# LABEL_COL = "label"

if TEXT_COL is None or LABEL_COL is None:
    print("\nAvailable columns:", df.columns.tolist())
    print("Please manually set TEXT_COL and LABEL_COL based on your dataset.")

## 3. Dataset Information

In [ ]:
if TEXT_COL and LABEL_COL:
    print("Dataset shape:", df.shape)
    print("\nDataset information:")
    df.info()

In [ ]:
if TEXT_COL and LABEL_COL:
    print("Missing values:")
    display(df[[TEXT_COL, LABEL_COL]].isnull().sum().to_frame("Missing Values"))

    print("\nDuplicate rows:", df.duplicated().sum())

## 4. Clean Basic Records

Remove rows where the message or target label is missing and remove duplicate records.

In [ ]:
if TEXT_COL and LABEL_COL:
    before = len(df)

    df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
    df = df.drop_duplicates().copy()

    after = len(df)

    print("Rows before cleaning:", before)
    print("Rows after cleaning:", after)
    print("Rows removed:", before - after)

## 5. Spam vs Ham Distribution

In [ ]:
if LABEL_COL:
    print("Class distribution:")
    display(df[LABEL_COL].value_counts())

    print("\nClass percentages:")
    display((df[LABEL_COL].value_counts(normalize=True) * 100).round(2))

In [ ]:
if LABEL_COL:
    plt.figure(figsize=(7, 5))
    df[LABEL_COL].value_counts().plot(kind="bar")
    plt.title("Spam vs Ham Email Distribution")
    plt.xlabel("Email Class")
    plt.ylabel("Number of Emails")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 6. Message Length Analysis

Message length is calculated as the number of characters in each email/message.

In [ ]:
if TEXT_COL:
    df["message_length"] = df[TEXT_COL].astype(str).str.len()

    display(df["message_length"].describe())

In [ ]:
if TEXT_COL:
    plt.figure(figsize=(8, 5))
    plt.hist(df["message_length"], bins=40)
    plt.title("Distribution of Email Message Length")
    plt.xlabel("Message Length (characters)")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

In [ ]:
if TEXT_COL and LABEL_COL:
    plt.figure(figsize=(8, 5))
    df.boxplot(column="message_length", by=LABEL_COL)
    plt.title("Message Length by Email Class")
    plt.suptitle("")
    plt.xlabel("Email Class")
    plt.ylabel("Message Length")
    plt.tight_layout()
    plt.show()

## 7. Word Count Analysis

In [ ]:
if TEXT_COL:
    df["word_count"] = df[TEXT_COL].astype(str).apply(lambda x: len(x.split()))

    display(df["word_count"].describe())

In [ ]:
if TEXT_COL and LABEL_COL:
    plt.figure(figsize=(8, 5))
    df.boxplot(column="word_count", by=LABEL_COL)
    plt.title("Word Count by Email Class")
    plt.suptitle("")
    plt.xlabel("Email Class")
    plt.ylabel("Number of Words")
    plt.tight_layout()
    plt.show()

## 8. Most Frequent Words

This section performs a simple frequency analysis after converting text to lowercase and removing punctuation.

In [ ]:
from collections import Counter

def get_words(series):
    text = " ".join(series.astype(str)).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    words = text.split()
    return words

if TEXT_COL:
    all_words = get_words(df[TEXT_COL])
    word_counts = Counter(all_words)

    print("Top 20 words in the complete dataset:")
    display(pd.DataFrame(word_counts.most_common(20), columns=["Word", "Frequency"]))

In [ ]:
if TEXT_COL and LABEL_COL:
    classes = df[LABEL_COL].astype(str).unique()

    for cls in classes:
        class_words = get_words(df.loc[df[LABEL_COL].astype(str) == cls, TEXT_COL])
        counts = Counter(class_words)

        print(f"\nTop 15 words for class: {cls}")
        display(pd.DataFrame(counts.most_common(15), columns=["Word", "Frequency"]))

## 9. EDA Observations

After executing the notebook, record the actual observations from your dataset below.

Example observations to replace with your actual findings:

- The dataset contains both Spam and Ham email messages.
- The class distribution shows whether the dataset is balanced or imbalanced.
- Email/message lengths vary considerably between records.
- Spam and Ham messages may show different text-length and word-frequency patterns.
- Frequent-word analysis helps identify common terms associated with different email categories.

In [ ]:
# Print a compact summary for the Review 3 discussion
if TEXT_COL and LABEL_COL:
    print("FINAL EDA SUMMARY")
    print("-" * 50)
    print("Total records:", len(df))
    print("Text column:", TEXT_COL)
    print("Label column:", LABEL_COL)
    print("Number of classes:", df[LABEL_COL].nunique())
    print("\nClass counts:")
    print(df[LABEL_COL].value_counts())
    print("\nAverage message length:", round(df["message_length"].mean(), 2))
    print("Average word count:", round(df["word_count"].mean(), 2))

## 10. Conclusion

The EDA provides an understanding of the Spam Email Classification dataset, its class distribution, message characteristics, and frequently occurring words.

The next stage of the project is to perform text preprocessing and feature extraction (such as TF-IDF), train a classification algorithm, and evaluate its performance using accuracy, precision, recall, F1-score, and a confusion matrix.